# Artikkeloversikt

Henter strukturert data fra SQLite (elementer, sammendrag, evalueringstriplets) og eksporterer til Excel.

In [5]:
import sqlite3, os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display

repo_rot = Path("__file__").resolve().parent.parent
load_dotenv(repo_rot / ".env")
DB_STI = repo_rot / os.getenv("DATABASE_STI", "data/monitor.db")

print(f"Database : {DB_STI}  (finnes: {DB_STI.exists()})")

Database : C:\Users\Audun\2026\claude\personalisert_monitor\data\monitor.db  (finnes: True)


In [6]:
SQL = """
SELECT
    e.id            AS element_id,
    k.navn          AS kilde,
    e.tittel,
    e.publisert,
    e.url,
    s.tekst         AS sammendrag,
    s.prompt_versjon,
    t.godkjent,
    t.kommentar     AS ekspert_kommentar,
    t.tidsstempel   AS vurdert
FROM elementer e
JOIN kilder k ON e.kilde_id = k.id
LEFT JOIN sammendrag s ON s.element_id = e.id
LEFT JOIN evalueringstriplets t
    ON t.element_id = e.id AND t.komponent = 'sammendrag'
WHERE e.dead_letter = 0
ORDER BY e.publisert DESC
"""

with sqlite3.connect(DB_STI) as conn:
    df = pd.read_sql_query(SQL, conn)

df["godkjent"] = df["godkjent"].map({1: "Godkjent", 0: "Avvist"}).fillna("Ikke vurdert")

print(f"Rader: {len(df)}  |  Kolonner: {list(df.columns)}")

Rader: 4  |  Kolonner: ['element_id', 'kilde', 'tittel', 'publisert', 'url', 'sammendrag', 'prompt_versjon', 'godkjent', 'ekspert_kommentar', 'vurdert']


In [7]:
pd.set_option("display.max_colwidth", None)

display(
    df.style
    .set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
)

,element_id,kilde,tittel,publisert,url,sammendrag,prompt_versjon,godkjent,ekspert_kommentar,vurdert
0,49,manuell-klipp,Effective context engineering for AI agents,None,https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents,"Effektiv kontekststyring for AI-agenter: Fra prompt til dynamisk kontekst Oppsummering Bygging av AI-agenter har beveget seg fra å handle om å skrive gode instruksjoner (prompt engineering) til å optimalisere hvilke data og instruksjoner som faktisk sendes inn til språkmodellen i hvert kall (kontekst engineering). Utfordringen er at språkmodeller har begrenset oppmerksomhetskapasitet: for mye eller feil sammensatt kontekst gir svekket ytelse, tap av fokus og vanskelig feilsøking. Effektiv kontekststyring handler om å velge ut et minimalt, men tilstrekkelig informativt sett med tokens for hver modellkjøring, og å bygge systemer som dynamisk henter inn relevant informasjon etter behov. Hva er nytt 1. Kontekst engineering som iterativ prosess I stedet for å kun fokusere på å skrive gode systemprompter, handler kontekst engineering om å kuratere hele informasjonsgrunnlaget for hvert enkelt kall til modellen. Dette inkluderer systeminstruksjoner, relevante eksempler, verktøybeskrivelser, tidligere meldinger og eksterne data. I praksis betyr dette at systemet må ha en løpende mekanisme for å velge hvilke data som skal med i konteksten basert på oppgaven og tidligere interaksjoner. For eksempel: en agent som jobber over flere steg, må etter hvert filtrere ut irrelevante meldinger og kun ta med det som faktisk påvirker neste beslutning. 2. Dynamisk og selektiv innhenting av kontekst Moderne agent-arkitekturer beveger seg bort fra å laste inn all mulig relevant informasjon på forhånd. I stedet brukes lette referanser (som filbaner eller søkestrenger) og verktøy for å hente inn data akkurat når det trengs (""just in time""). Et konkret scenario: en kodeagent får kun filnavn og katalogstruktur i starten, og bruker deretter søk eller spørringer for å hente inn kodebiter eller dokumentasjon etter behov. Dette gir lavere tokenbruk, raskere iterasjon og mindre risiko for at modellen mister fokus. 3. Strategier for langvarige og komplekse oppgaver For oppgaver som varer lenger enn kontekstvinduet tillater, brukes teknikker som kompaktering (summarization av tidligere samtale eller arbeidslogg), strukturert notattaking og multi-agent-arkitekturer. Eksempel: en agent som migrerer en stor kodebase, oppsummerer løpende hva som er gjort og hvilke beslutninger som er tatt, og starter nye kontekstvinduer med disse oppsummeringene som grunnlag. Dette gjør det mulig å opprettholde sammenheng og målrettethet over tid uten å drukne modellen i gamle detaljer. Hvorfor det er viktig Kontekst engineering gjør det mulig å bygge agenter som faktisk holder seg på sporet, gir forutsigbare resultater og kan feilsøkes systematisk. Det reduserer risikoen for at systemet feiler stille eller gir uforståelige svar fordi relevant informasjon er druknet i støy. For ingeniører betyr dette bedre kontrollpunkter, lavere driftskostnader per kall, og raskere iterasjon når systemet skal forbedres eller feilsøkes. Regulatorisk kontekst Ingen direkte regulatoriske krav adresseres i artikkelen, men kontekst engineering påvirker hvordan systemer kan dokumentere hvilke data som faktisk ble brukt i hver modellkjøring. Dette er relevant for etterlevelse av krav til revisjonsspor og ansvarlighet i AI-systemer, særlig i høyrisiko-domener under AI Act og for dokumentasjonskrav i ISO 42001.",v1,Ikke vurdert,None,None
1,50,manuell-klipp,"See What Your AI Sees: Multimodal Tracing for Images, Audio, and Files",None,https://mlflow.org/blog/multimodal-tracing/,"Multimodal sporing gir bedre innsikt i AI-systemers bilde-, lyd- og filbehandling Oppsummering Når AI-systemer håndterer bilder, lyd eller dokumenter, blir feilsøking vanskelig hvis sporingsdata kun viser base64-strenger uten faktisk innhold. Dette gir både lagringsproblemer og gjør det 

In [8]:
from datetime import date

eksport_mappe = repo_rot / "data" / "eksport"
eksport_mappe.mkdir(parents=True, exist_ok=True)

filnavn = eksport_mappe / f"artikkeloversikt_{date.today()}.xlsx"
df.to_excel(filnavn, index=False, engine="openpyxl")

print(f"Lagret: {filnavn.resolve()}")

Lagret: C:\Users\Audun\2026\claude\personalisert_monitor\data\eksport\artikkeloversikt_2026-05-05.xlsx
